# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Metadata
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', 'Dataset')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets and fields, including their `@id` values. All references below use the `@id`.

Let's look at the available RecordSets, their fields, and columns.


In [ ]:
# List all available record sets

print("=== RecordSets ===")
record_set_ids = []
if hasattr(metadata, 'record_set'):
    for rs in getattr(metadata, 'record_set'):
        print(f"@id: {getattr(rs, '@id', '')}, name: {getattr(rs, 'name', '')}")
        record_set_ids.append(getattr(rs, '@id', ''))
        # Show fields and columns
        if hasattr(rs, 'field'):
            print('  Fields:')
            for field in rs.field:
                print(f"    @id: {getattr(field, '@id', '')}, name: {getattr(field, 'name', '')}")
                if hasattr(field, 'column'):
                    print("      Columns:")
                    for col in (field.column if isinstance(field.column, list) else [field.column]):
                        print(f"       @id: {getattr(col, '@id', '')}, name: {getattr(col, 'name', '')}")
else:
    print("No record sets found in the provided metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Below, we will use the RecordSet and field `@id`s obtained above.

**Tip:** Replace `<record_set_id>` with one of the RecordSet `@id`s found above.


In [ ]:
# Define your record set(s). For this dataset, get all available record set ids (found above).
# If none were found, you may need to directly inspect the metadata or adjust for legacy Croissant versions.

if not record_set_ids:
    # If no record_set found, try extracting from raw JSON
    meta_json = dataset.metadata.to_json() if hasattr(dataset.metadata, 'to_json') else dataset.metadata
    if isinstance(meta_json, dict) and 'recordSet' in meta_json:
        record_set_ids = [rs['@id'] for rs in meta_json['recordSet']]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns and head for first record set if exists
if record_set_ids:
    rs_to_show = record_set_ids[0]
    print(f"Columns for record set {rs_to_show}:")
    print(dataframes[rs_to_show].columns.tolist())
    display(dataframes[rs_to_show].head())
else:
    print("No record sets found for data extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, e.g., filtering, normalizing, grouping.

Let's select a numeric field from the extracted DataFrame. You can check the column list printed above to choose which field to analyze.

In [ ]:
import numpy as np

# Select record set for EDA
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id] if record_set_id else pd.DataFrame()

# Inspect numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist() if not df.empty else []
print("Numeric columns:", numeric_cols)

if numeric_cols:
    # Choose the first numeric field for filtering
    numeric_field = numeric_cols[0]
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (showing first 5):")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if possible
    group_fields = df.select_dtypes(include=[object]).columns.tolist()
    group_field = group_fields[0] if group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field} (first 5 groups):")
        print(grouped_df.head())
else:
    print("No numeric columns available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is a sample visualization of the first numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_cols:
    field = numeric_cols[0]
    plt.figure(figsize=(8,4))
    sns.histplot(df[field], kde=True)
    plt.title(f"Distribution of {field}")
    plt.xlabel(field)
    plt.ylabel('Count')
    plt.show()

    # If there's a group_field, show boxplot
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=field, data=df)
        plt.title(f"{field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric data available for plotting.")

## 6. Conclusion
In this notebook, we loaded the Mass Spectrometry dataset using `mlcroissant`, explored its structure, fields, and records, performed basic EDA including outlier filtering and normalization, and visualized a numeric field. For full analysis, select domain-specific fields and apply relevant methods based on your research question.
